# Gh0st in the L00p — Full Pipeline

One notebook: background removal → dataset prep → sequence optimisation → manual adjustment → video.

**Setup:** Run cell 1 (pip install), then **Runtime → Restart session**, then continue from cell 2.

In [ ]:
# Run once, then restart runtime before continuing.
!pip install 'rembg[cpu]' ultralytics gdown opencv-python-headless imageio imageio-ffmpeg -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
repo_dir = '/content/gh0st-in-the-l00p'
if os.path.exists(repo_dir):
    !git -C {repo_dir} pull -q
else:
    !git clone https://github.com/jasonr2048/gh0st-in-the-l00p.git {repo_dir} -q
%cd {repo_dir}
print('Ready.')

---
## CONFIG
**Edit this cell only.** Everything else derives from it.

In [ ]:
# ── Sets ──────────────────────────────────────────────────────────────────────
# Ordered list of style sets to include in the video.
set_names = [
    'rhinestones',
    'gold_hardware',
    'blue_face',
    'clown',
    'crybaby',
    'horror_drip',
    'latex_skin',
    'ceramic_mask',
]

# Sets that need background removal before prep (from raw/tricky/).
# Leave empty if all sets are in raw/ready/.
bg_removal_sets = []  # e.g. ['nat_indian', 'smiley_holo', 'camo_studio']

# ── Prep params ───────────────────────────────────────────────────────────────
resolution  = 512    # output image size (square)
scale       = 2.5    # crop size as multiple of head keypoint span
mouth_scale = 0.8    # mouth offset as multiple of nose-to-eye/ear distance

# ── Interpolation params ──────────────────────────────────────────────────────
fps            = 24
hold_frames    = 12   # frames to hold on each image before morphing
morph_frames   = 24   # frames for morph between images within a set
between_frames = 48   # frames for morph between the last image of one set and first of the next

# ── Paths ─────────────────────────────────────────────────────────────────────
drive_root      = '/content/drive/MyDrive/Gh0st in the Loop'
raw_ready_root  = f'{drive_root}/dataset/raw/ready'
raw_tricky_root = f'{drive_root}/dataset/raw/tricky'
bg_removed_root = f'{drive_root}/dataset/bg_removed'
prepared_root   = f'{drive_root}/dataset/prepared_{resolution}px_s{scale}_m{mouth_scale}'
output_dir      = f'{drive_root}/outputs'
output_path     = f'{output_dir}/interpolation_flow.mp4'
seq_path        = f'{output_dir}/sequence.json'
manual_path     = f'{output_dir}/manual_sequence.json'
face_model_path = f'{drive_root}/models/yolov8n-face.pt'

os.makedirs(output_dir, exist_ok=True)
print(f'Prepared dir : {prepared_root}')
print(f'Output video : {output_path}')

---
## A — Background Removal
Only runs for sets listed in `bg_removal_sets`. Skips images already in `bg_removed/`.

In [ ]:
from pathlib import Path
from PIL import Image

if not bg_removal_sets:
    print('No bg_removal_sets configured — skipping.')
else:
    from rembg import remove
    SUPPORTED = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tiff'}

    for set_name in bg_removal_sets:
        src_dir = Path(raw_tricky_root) / set_name
        out_dir = Path(bg_removed_root) / set_name
        out_dir.mkdir(parents=True, exist_ok=True)

        images = [p for p in src_dir.rglob('*') if p.suffix.lower() in SUPPORTED]
        print(f'\n{set_name} — {len(images)} images')

        for i, path in enumerate(images, 1):
            out_path = out_dir / path.name
            if out_path.exists():
                print(f'  [{i}/{len(images)}] {path.name} ... skipped')
                continue
            try:
                img = Image.open(path).convert('RGBA')
                result = remove(img)
                bg = Image.new('RGB', img.size, (0, 0, 0))
                bg.paste(result.convert('RGB'), mask=result.split()[3])
                bg.save(out_path)
                print(f'  [{i}/{len(images)}] {path.name} ... ok')
            except Exception as e:
                print(f'  [{i}/{len(images)}] {path.name} ... ERROR: {e}')

    print('\nBackground removal done.')

---
## B — Dataset Prep

Crops and resizes each set to `{resolution}x{resolution}`. Output folder name encodes the config,
so changing `scale`, `mouth_scale`, or `resolution` automatically creates a fresh folder.
Skips sets whose output subfolder already exists and is non-empty.

In [ ]:
import subprocess
from pathlib import Path

for set_name in set_names:
    out_subdir = Path(prepared_root) / set_name

    # Skip if already prepared (non-empty folder)
    if out_subdir.exists() and any(out_subdir.iterdir()):
        n = len(list(out_subdir.glob('*.png')))
        print(f'{set_name}: already prepared ({n} images) — skipping')
        continue

    # Source: bg_removed if this set needed bg removal, else raw/ready
    if set_name in bg_removal_sets:
        src = f'{bg_removed_root}/{set_name}'
    else:
        src = f'{raw_ready_root}/{set_name}'

    print(f'\nPreparing {set_name} from {src} ...')
    result = subprocess.run(
        [
            'python', 'tools/prepare_dataset_v2.py',
            '--input_dir',       src,
            '--output_dir',      str(out_subdir),
            '--resolution',      str(resolution),
            '--scale',           str(scale),
            '--mouth_scale',     str(mouth_scale),
            '--face_model_path', face_model_path,
        ],
        capture_output=False,
    )
    if result.returncode != 0:
        print(f'  ERROR: prepare_dataset_v2 exited with code {result.returncode}')

print('\nDataset prep complete.')

---
## C — Sequence Optimisation

Orders images within each set using 2-opt, then chooses forward/reverse across sets.
Saves `sequence.json` and displays a visualisation.

In [ ]:
import subprocess
from PIL import Image as PILImage
from IPython.display import display

result = subprocess.run(
    [
        'python', 'spikes/optical_flow_interpolation/optimise_sequence.py',
        '--prepared_dir', prepared_root,
        '--sets',         ','.join(set_names),
        '--output',       seq_path,
    ],
    capture_output=True, text=True,
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr)

vis_path = seq_path.replace('.json', '.png')
if os.path.exists(vis_path):
    display(PILImage.open(vis_path))

---
## D — Review & Manual Adjustment

**Iterative section.** Run the grid cell to see current order, edit `manual_order`, run the save cell,
then re-run the grid cell to verify. Repeat until happy, then continue to Section E.

- Numbers in the grid are the indices to use in `manual_order`.
- Leave `[]` to keep the auto-optimised order for that set.
- Repeat an index to use that image more than once (e.g. `[1, 2, 1, 3]`).

In [ ]:
# Show image grids in current effective order (auto + any manual overrides).
# Re-run this cell after saving manual overrides to verify.

import json
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display

with open(seq_path) as f:
    sequence = json.load(f)

if os.path.exists(manual_path):
    with open(manual_path) as f:
        manual = json.load(f)
    manual_map = {s['name']: s['images'] for s in manual['sets']}
    for s in sequence['sets']:
        if s['name'] in manual_map:
            s['images'] = manual_map[s['name']]
    print(f'Manual overrides active for: {list(manual_map.keys())}')
else:
    print('No manual overrides — showing auto-optimised order.')

THUMB = 160
for s in sequence['sets']:
    imgs = s['images']
    n = len(imgs)
    grid = Image.new('RGB', (n * THUMB, THUMB + 24), (30, 30, 30))
    draw = ImageDraw.Draw(grid)
    for i, img_path in enumerate(imgs):
        try:
            thumb = ImageOps.exif_transpose(Image.open(img_path).convert('RGB')).resize((THUMB, THUMB))
        except Exception:
            thumb = Image.new('RGB', (THUMB, THUMB), (80, 0, 0))
        grid.paste(thumb, (i * THUMB, 0))
        draw.text((i * THUMB + THUMB // 2, THUMB + 4), str(i + 1),
                  fill=(220, 220, 220), anchor='mt')
    print(f'\n{s["name"]} ({n} images):')
    display(grid)

In [ ]:
# ── EDIT THIS CELL ────────────────────────────────────────────────────────────
# Use 1-based indices from the grids above.
# [] = keep auto order.  Repeats allowed, e.g. [1, 2, 1, 3].
# ─────────────────────────────────────────────────────────────────────────────
manual_order = {
    'rhinestones':   [],
    'gold_hardware': [],
    'blue_face':     [],
    'clown':         [],
    'crybaby':       [],
    'horror_drip':   [],
    'latex_skin':    [],
    'ceramic_mask':  [],
}

In [ ]:
# Save manual overrides, then re-run the grid cell above to verify.

import json

with open(seq_path) as f:
    auto_seq = {s['name']: s['images'] for s in json.load(f)['sets']}

manual_sets = []
for name, order in manual_order.items():
    if not order:
        continue
    imgs = auto_seq.get(name, [])
    if not imgs:
        print(f'  WARNING: {name} not found in auto sequence — skipping')
        continue
    try:
        ordered_paths = [imgs[i - 1] for i in order]
    except IndexError as e:
        print(f'  ERROR in {name}: {e} (max index is {len(imgs)})')
        continue
    manual_sets.append({'name': name, 'images': ordered_paths})
    print(f'  {name}: {order}')

if manual_sets:
    with open(manual_path, 'w') as f:
        json.dump({'sets': manual_sets}, f, indent=2)
    print(f'\nSaved overrides for {len(manual_sets)} set(s) → {manual_path}')
    print('Re-run the grid cell to verify.')
else:
    if os.path.exists(manual_path):
        os.remove(manual_path)
        print('All entries are [] — removed manual_sequence.json (using auto order).')
    else:
        print('No overrides — using auto order.')

---
## E — Generate Video

Builds frames using optical flow interpolation. Each `{resolution}×{resolution}` face is
centred on a `1080×1920` portrait canvas. Flow is computed at face resolution for speed.

Run section C first if you've changed `set_names` since the last run.

In [ ]:
import json, cv2, numpy as np
from PIL import Image, ImageOps

# ── Load effective sequence (auto + manual overrides) ─────────────────────────
with open(seq_path) as f:
    sequence = json.load(f)

if os.path.exists(manual_path):
    with open(manual_path) as f:
        manual = json.load(f)
    manual_map = {s['name']: s['images'] for s in manual['sets']}
    for s in sequence['sets']:
        if s['name'] in manual_map:
            s['images'] = manual_map[s['name']]
    print(f'Manual overrides applied: {list(manual_map.keys())}')

all_images = [(img_path, s['name'])
              for s in sequence['sets']
              for img_path in s['images']]
print(f'Total images in sequence: {len(all_images)}')
for s in sequence['sets']:
    print(f'  {s["name"]}: {len(s["images"])} images')

# ── Canvas helpers ────────────────────────────────────────────────────────────
CANVAS_W, CANVAS_H = 1080, 1920
face_size = resolution
x_off = (CANVAS_W - face_size) // 2
y_off = (CANVAS_H - face_size) // 2

def place_on_canvas(pil_face):
    """Centre a face_size×face_size image on a 1080×1920 black canvas."""
    canvas = Image.new('RGB', (CANVAS_W, CANVAS_H), (0, 0, 0))
    canvas.paste(pil_face.resize((face_size, face_size), Image.LANCZOS), (x_off, y_off))
    return canvas

def optical_flow_morph(img_a, img_b, steps):
    """Morph at face_size resolution; return portrait-canvas frames."""
    sz = (face_size, face_size)
    a = cv2.resize(np.array(img_a.convert('RGB')), sz)
    b = cv2.resize(np.array(img_b.convert('RGB')), sz)
    a_g = cv2.cvtColor(a, cv2.COLOR_RGB2GRAY)
    b_g = cv2.cvtColor(b, cv2.COLOR_RGB2GRAY)
    flow = cv2.calcOpticalFlowFarneback(a_g, b_g, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    h, w = a.shape[:2]
    bx = np.tile(np.arange(w), (h, 1)).astype(np.float32)
    by = np.tile(np.arange(h), (w, 1)).T.astype(np.float32)
    frames = []
    for t in np.linspace(0, 1, steps):
        warped = cv2.remap(a, (bx + flow[..., 0] * t).astype(np.float32),
                              (by + flow[..., 1] * t).astype(np.float32),
                              cv2.INTER_LINEAR)
        blended = cv2.addWeighted(warped, 1 - t, b, t, 0)
        frames.append(place_on_canvas(Image.fromarray(blended)))
    return frames

def hold(img, n):
    return [place_on_canvas(img)] * n

# ── Build frames ──────────────────────────────────────────────────────────────
all_frames = []
n = len(all_images)

for i, (img_path, set_name) in enumerate(all_images):
    img = ImageOps.exif_transpose(Image.open(img_path).convert('RGB'))
    next_img_path, next_set = all_images[(i + 1) % n]
    next_img = ImageOps.exif_transpose(Image.open(next_img_path).convert('RGB'))

    if hold_frames > 0:
        all_frames.extend(hold(img, hold_frames))

    is_set_boundary = (next_set != set_name)
    n_morph = between_frames if is_set_boundary else morph_frames
    all_frames.extend(optical_flow_morph(img, next_img, n_morph))

    if (i + 1) % 5 == 0 or i + 1 == n:
        print(f'  [{i + 1}/{n}] {len(all_frames)} frames so far...', end='\r')

total_s = len(all_frames) / fps
print(f'\nTotal frames: {len(all_frames)} ({total_s:.1f}s at {fps}fps)')

In [ ]:
# Preview: 10 evenly-spaced frames as a contact sheet.
import numpy as np
from PIL import Image
from IPython.display import display

PREVIEW_N = 10
thumb_w, thumb_h = 108, 192
indices = np.linspace(0, len(all_frames) - 1, PREVIEW_N, dtype=int)
sheet = Image.new('RGB', (PREVIEW_N * thumb_w, thumb_h))
for j, idx in enumerate(indices):
    sheet.paste(all_frames[idx].resize((thumb_w, thumb_h)), (j * thumb_w, 0))
display(sheet)
print(f'{PREVIEW_N} evenly-spaced frames from {len(all_frames)} total')

In [ ]:
# Save video to Drive.
import imageio, numpy as np

with imageio.get_writer(output_path, fps=fps) as writer:
    for i, frame in enumerate(all_frames):
        writer.append_data(np.array(frame))
        if i % 100 == 0:
            print(f'  {i}/{len(all_frames)} frames written...', end='\r')

size_mb = os.path.getsize(output_path) / 1e6
print(f'\nVideo saved: {output_path}  ({size_mb:.1f} MB, {total_s:.1f}s)')